# 一：RAG

---

## 1.1 RAG（检索增强生成）

**定义**：引入外部知识库，让大模型在回答前先在这个外部知识库中检索，再回答

### 优点 ✅

- 回答准确，检索库可以更新，所以回答的依据也可以更新
- 可解释性强，可以在检索库核实准确性
- 缓解幻觉问题（避免胡说八道）
- 在本地，隐私性好

### RAG 对比 模型微调

| 特性 | RAG | 模型微调 |
|------|-----|----------|
| 知识更新 | 动态更新 | 需要重新训练 |
| 定制化程度 | 低 | 高 |
| 数据处理需求 | 低 | 高 |
| 幻觉问题 | 缓解 | 减少 |
| 计算资源 | 需要检索 | 需要训练 |
| 可追溯性 | 可追溯 | 难追溯 |

---

## 1.1.1 基本步骤

分为三大步骤：**建库 -> 检索 -> 生成**（`indexing` -> `retrieval` -> `generation`）

### 建库（Indexing）🔧

| 步骤 | 操作 | 说明 |
|------|------|------|
| 1 | 收集数据 | 收集原始数据，转化为统一文本格式 |
| 2 | 文本分割 | 将文本分割成适当大小的块（chunk） |
| 3 | 向量化 | 使用预训练模型将文本转化为向量 |
| 4 | 存储 | 将向量存入向量数据库，构建索引 |

### 检索（Retrieval）🔍

| 步骤 | 操作 | 说明 |
|------|------|------|
| 1 | Query Translation | 查询改写：Multi Query、RAG Fusion、Decomposition、Step Back、HyDE |
| 2 | Routing | 路由：根据 query 分类，选择正确的数据源 |
| 3 | Query Construction | 查询构造：NER、关键词提取，转为数据库查询参数 |
| 4 | 向量检索 | 计算相似度，找出最相关的文档 |
| 5 | 排序/重排 | 根据相似度排序，优化结果顺序 |
| 6 | 过滤 | 过滤无关或低质量结果 |

### 生成（Generation）📝

| 步骤 | 操作 | 说明 |
|------|------|------|
| 1 | 构建 Prompt | 将 query 和检索到的文档结合 |
| 2 | LLM 生成 | 基于文档生成答案 |
| 3 | 后处理 | 置信度评估、格式解析、多候选筛选 |

---

## 1.1.2 Query Translation（查询改写）🔄

**属于检索阶段**，将 question 变成更容易检索的形式，提升后续检索的效果

---

### 1. Multi Query（多查询）

将 question 从**多个角度**改写，每一个 question 都进行检索，查询结果取**并集**

- 可以获得更全面的文档集合
- 缺点：可能引入过多不相关内容

---

### 2. RAG Fusion（RAG 融合）

对于检索到的文档做**相互排名**，通过加权各个 question 在文档中的查询结果，得到**综合排名**

- 取前几名作为最终结果
- 比 Multi Query 更精确

---

### 3. Decomposition（查询分解）

将原始 query 拆解成**多个子 query**，每个子 query 都进行检索

| 方式 | 说明 |
|------|------|
| **顺序依赖** | 每个子问题会影响后续子问题的提问和解答过程，类似局部推理 |
| **独立并行** | 每个子问题互不影响，最后合并答案 |

---

### 4. Step Back（后退）

从一个**具体问题**出发，通过 **few-shot**，生成一个更高层次、更抽象的问题

- 抽象问题有助于检索到更相关的基础概念
- 适用于：原始问题过于细节，难以直接检索的情况

---

### 5. HyDE（Hypothetical Document Embeddings）📄

**假设文档嵌入**，核心思想：

1. 利用已有的大模型生成一个**假设性回答**
2. 将生成的假设文档输入编码器生成**嵌入向量**
3. 用这个向量去找**语义相似**的真实文档

**优势**：假设文档中包含相关关键词，可以检索到更多相关内容

---

## 总结：Query Translation 方法对比

| 方法 | 核心思想 | 适用场景 |
|------|----------|----------|
| Multi Query | 多角度改写 | 需要全面覆盖 |
| RAG Fusion | 综合排名 | 平衡全面与精确 |
| Decomposition | 拆解子问题 | 复杂多部分问题 |
| Step Back | 抽象化 | 问题过于细节 |
| HyDE | 假设文档 | 关键词匹配困难 |

---

## 1.1.3 Routing（路由）🎯

**属于检索阶段**，本质是一个**分类任务**，根据 question 判断去哪个数据源检索

---

### 实现方式

| 方式 | 说明 | 优点 | 缺点 |
|------|------|------|------|
| **基于规则** | 用预定义关键词匹配 | 简单快速 | 关键词覆盖不全 |
| **基于机器学习** | 收集样本，微调模型 | 准确率高 | 需要训练数据 |
| **基于提示词工程** | 用大模型判断意图 | 零成本/少样本 | 依赖模型能力 |

---

### 路由类型

| 类型 | 说明 |
|------|------|
| **Logical Routing** | 将 query 路由到最相关的资源库 |
| **Semantic Routing** | 用 embeddings 找最相关的 prompt 或文档 |

---

## 1.1.4 Query Construction（查询构造）🔧

**属于检索阶段**，把用户的**自然语言**转换成**数据库能理解的查询参数**

---

### 核心思想

利用**命名实体识别（NER）**等技术，将自然语言 question 转成合适的搜索和过滤参数

---

### 示例

**用户 query（自然语言）**：
```
how to use multi-modal models in an agent, only videos under 5 minutes
```

**转换成搜索参数**：

| 参数 | 值 | 含义 |
|------|-----|------|
| content_search | multi-modal models agent | 搜索内容 |
| title_search | multi-modal models agent | 搜索标题 |
| max_length_sec | 300 | 只找 5 分钟以下的视频 |

---

### 适用场景

- SQL 数据库查询
- Elasticsearch 搜索
- API 参数构造
- 任何需要结构化查询的场景

---

## 1.1.5 Indexing（索引）📚

**属于建库阶段**，将文档拆成向量（Vector）形式，建立索引

---

### 1. Chunking（分块）

**定义**：将大块文本分解成小段（chunk）的过程

**影响**：

| 方面 | 说明 |
|------|------|
| 检索精度 | 合理的切分保证每个 chunk 的语义完整性，更准确匹配用户意图 |
| 向量表示 | 产生更精确的语义向量，使相似度计算更准确，加速索引查找 |
| 生成质量 | 在不超过模型输入限制的情况下提供足够上下文，减轻幻觉 |
| 实际应用 | 不同领域有不同策略，保留元数据可提升检索效果 |

---

### 现有分块方案

| 方案 | 说明 |
|------|------|
| FastGPT | 快速分块工具 |
| LangChain-Chat | LangChain 的分块模块 |
| Baidu 千帆知识库 | 百度知识库方案 |
| 星火知识库 | 科大讯飞方案 |
| Jina | Jina AI 的分块工具 |

---

### 基于 LangChain 的分块方法

| 方法 | 说明 |
|------|------|
| **Character** | 按固定字符数分割，可设置重叠字符保持上下文 |
| **Recursive** | 默认方法，按不同字符递归分割（`["\n\n", "\n", " ", ""]`），实现结构化切分 |
| **Token** | 按 token 数量分块，常用分词器：BPE、tiktoken |
| **Document** | 使用特定分隔符（如 Markdown 标题、Python 类/函数） |
| **Semantic** | 计算句子间 embedding 距离，相似主题分为一块 |
| **Agentic** | 最高级方法，通过 LLM 做决策，将文本分块为独立命题 |

---

### 分块时考虑的因素

| 因素 | 说明 |
|------|------|
| **内容性质** | 长文档（文章/书籍）vs 短内容（微博/消息） |
| **Embedding 模型** | sentence-transformer 适合单句；text-embedding-ada-002 适合 256-512 tokens |
| **查询期望** | 简短具体问题 vs 冗长复杂问题，影响分组方式 |
| **检索结果用途** | 语义搜索、问答、摘要等不同目的 |

---

### 文档解析与 OCR 改进

**统一解析模块**：针对不同格式分别处理

| 格式 | 处理方法 |
|------|----------|
| **PDF** | 优先文本层提取（pdfplumber/PyMuPDF）；扫描 PDF 用 OCR（Tesseract/PaddleOCR）；表格用专门处理；推荐 Marker 和 MinerU |
| **PPT** | 利用幻灯片结构提取标题和文本框；图片用 OCR 提取文字 |
| **纯文本** | 按行/段读取，识别 Markdown 代码块、表格等特殊标记 |
| **视频** | 语音识别（ASR）得到字幕，按时间戳或语义合并成段落 |

---

### 智能 Chunk 切分（规则 + 语义融合）

**基于规则的切分**：
- 利用文档格式特征（章节标题、段落换行、列表项、表格边界）
- 表格和代码块整段视为一个 Chunk，避免中途截断

**语义连贯的调整**：
- 检查相邻 Chunks 的连贯性
- 过短且语义紧密的 Chunk 与相邻合并
- 跨页段落、跨页表格合并为整体

**长度和平衡**：
- 控制 Chunk 长度（如不超过 512 字或一定 token 数）
- 过长则按语义拆分，过短则与相邻补充

---

### 层级结构与标签管理

| 标签类型 | 说明 |
|----------|------|
| **章节层级** | 捕获文档层次（章节编号/标题），如"总则 > 范围"，检索时提高召回率 |
| **内容类别** | 标注类型（表格/代码块/普通文本）和主题（政策条例/操作指南），用于过滤或提高匹配度 |
| **引用与来源** | 记录文档名、页码、幻灯片编号，便于追溯原文和引用出处 |

---

### 2. Multi-Representation Indexing

**核心思想**：将用于答案生成的文档与用于检索的参考文档解耦

**方法流程**：
```
建库时：
1. 建立 docstore：存储完整文档
2. 利用 LLM 对文档做总结
3. 建立 vectorstore：存储总结的向量，用于检索

检索时：
1. question 与每个文档的总结进行相似度匹配
2. 根据匹配到的总结，在完整 docstore 中查询原文档
```

**优势**：
- 特别适用于**图像和表格**
- 解决了直接嵌入表格或图像（多模态嵌入）的挑战
- 使用总结作基于文本相似性搜索

---

### 3. RAPTOR（递归摘要树）

**问题**：典型 kNN 检索只适用于"低层次"query（单一文档可回答），不适用于"高层次"问题（需要多文档结合）

**核心思想**：通过创建**文档摘要**来捕捉更高层次概念

**步骤**：
1. 嵌入并聚类文档
2. 总结每个聚类
3. **递归**执行，产生包含越来越高层次概念的**摘要树**
4. 摘要和起始文档一起被索引，覆盖用户 query 的范围

**示意图**：
```
        根摘要（最高层次）
       /    |    \
   摘要 1  摘要 2  摘要 3
   / | \   / \   / | \
 文档 1... 文档 5... 文档 8...
```

---

### 4. ColBERT（上下文感知 Late Interaction）

**核心思想**：词级别的细粒度相似度计算

**通俗解释**：

| 方法 | 粒度 | 计算方式 |
|------|------|----------|
| **传统方法** | 文档级 | 把整个文档/问题变成一个向量，计算两个向量的相似度 |
| **ColBERT** | 词级别 | 把文档和问题都拆成一个个词（Tokens），每个词有自己的向量，词的向量之间计算相似度 |

---

**计算流程**：
```
问题："如何使用滑动窗口"
→ 拆成词：[如何，使用，滑动，窗口]
→ 每个词一个向量：[q1, q2, q3, q4]

文档："滑动窗口是一种算法技巧"
→ 拆成词：[滑动，窗口，是，一种，算法，技巧]
→ 每个词一个向量：[d1, d2, d3, d4, d5, d6]

计算相似度：
- q1("如何") 找文档中最相似的词 → max_sim(q1, 所有 d)
- q2("使用") 找文档中最相似的词 → max_sim(q2, 所有 d)
- q3("滑动") 找文档中最相似的词 → max_sim(q3, 所有 d)
- q4("窗口") 找文档中最相似的词 → max_sim(q4, 所有 d)

文档得分 = 所有 max_sim 的总和
```

---

**公式**：
```
Score(Q, D) = Σ_{q∈Q} max_{d∈D} Sim(q, d)

解释：
- Q: 问题的词向量集合
- D: 文档的词向量集合
- Sim(q, d): 词向量 q 和 d 的相似度
- 对问题中每个词，找文档中最相似的词，记录最大相似度
- 把所有词的最大相似度相加，得到文档总分
```

**优势**：

| 特性 | 说明 |
|------|------|
| **细粒度匹配** | 词级别匹配，比文档级别更精确 |
| **上下文感知** | 每个词的向量包含上下文信息 |
| **高效** | 可以预先计算文档侧，查询时只需计算问题侧 |
| **可解释** | 知道问题中哪些词匹配了文档中哪些词 |

---
